In [30]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import time
import os, json, ast, math, glob, time, copy
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
import glob
import pandas as pd
from sklearn.decomposition import PCA
import importlib

In [31]:
# ============================================================
# 1) PHYSICS / DOMAIN SETTINGS
# ============================================================
N = 24                        # 3D grid size along each axis (Nx = Ny = Nz = N)
Lx = Ly = Lz = 0.05            # domain lengths [m]

rho = 800.0                    # density [kg/m^3]
cp = 2000.0                    # specific heat [J/(kg.K)]
k = 0.2                        # thermal conductivity [W/(m.K)]
L_lat = 2e5                    # latent heat [J/kg]

T_m = 330.0                    # melting temperature [K]
T_init = 300.0                 # initial temperature [K]
T_bound = 330.0                # reference boundary temperature [K]

t_end = 4000.0                 # total simulation time [s]
save_times = (
    100.0, 250.0, 400.0, 600.0,
    1000.0, 1200.0, 1500.0, 1800.0, 2100.0
)                              # time snapshots saved from CFD
cfl = 0.30                    # CFL number for numerical stability


# ============================================================
# 2) VOLUMETRIC HEAT-SOURCE GENERATION
# ============================================================
q_scale = 1e5                  # heat-source magnitude scale [W/m^3]
Q_length_scale = 0.18          # GP/RBF length scale for source smoothness
Q_sigma = 1.0                  # GP/RBF amplitude scale


# ============================================================
# 3) BOUNDARY-CONDITION RANDOMIZATION
# ============================================================
mu_max = 0.8                   # max center-location parameter for BC profile
sigma_max = 0.7                # max spread parameter for BC profile
amp_max = 80                   # max BC amplitude variation

only_lr_vary = True            # if True, vary only left/right boundaries
All_side_const_temp_boundary = False
# if True, use constant temperature on all sides for testing/debugging


# ============================================================
# 4) DATASET SETTINGS
# ============================================================
NUM_CASES = 500                # total number of CFD/generated cases
TRAIN_FRAC = 0.8               # training split fraction
VAL_FRAC = 0.1                 # validation split fraction
# test fraction = 1 - TRAIN_FRAC - VAL_FRAC

TARGET = "f"                   # "T" for temperature, "f" for liquid fraction


# ============================================================
# 5) TRAINING SETTINGS
# ============================================================
BATCH_SIZE = 16
EPOCHS = 60
LR = 1e-3
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 0               # can increase later for faster loading
VAL_SAMPLES = 4                # number of validation cases/slices to visualize/check
WINDOW_SIZE = 13
WINDOW_RADIUS = WINDOW_SIZE // 2 
TIME_DIM = 64
TIME_N_FREQ = 8

# ------------------------------------------------------------
# LOSS WEIGHTS
# ------------------------------------------------------------
SMOOTHNESS_WEIGHT = 1e-4
GRAD_WEIGHT       = 0.15
INTERFACE_WEIGHT  = 0.2 if TARGET=="f" else 0  # used only if TARGET == "f"

INTERFACE_CENTER = 0.5
INTERFACE_SIGMA  = 0.15
INTERFACE_ALPHA  = 2.0

# ============================================================
# 6) UNIFIED 2D SLICE U-NET SETTINGS
# ============================================================
IN_CHANNELS = 18               # fixed number of input channels per 2D slice
UNET_FEATURES = 64             # base feature width
UNET_DROPOUT = 0.0
ACTIVATION_FUNCTION = "sin"    # keep only if your custom blocks actually use this


In [32]:
# ============================================================
# 0) DATASET RUN DIRECTORY + DEVICE SETUP
# ============================================================

ROOT = Path(".")   # parent folder containing dataset_run_* directories

def latest_run_dir(root: Path) -> Path | None:
    """Return the most recent dataset_run_* directory inside root."""
    run_dirs = sorted(
        [Path(p) for p in glob.glob(str(root / "dataset_run_*")) if Path(p).is_dir()]
    )
    return run_dirs[-1] if run_dirs else None


RUN_DIR = latest_run_dir(ROOT)
assert RUN_DIR is not None, "No dataset_run_* folder found. Generate the dataset first."

print(f"Using RUN_DIR: {RUN_DIR}")

# Main files expected inside RUN_DIR
DATASET_FILE = RUN_DIR / "slice_dataset_3d.npz"   # update this if your saved filename differs
SPLITS_FILE  = RUN_DIR / "splits.json"

assert DATASET_FILE.exists(), f"Dataset file not found: {DATASET_FILE}"
assert SPLITS_FILE.exists(),  f"Splits file not found: {SPLITS_FILE}"

# Device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"DEVICE: {DEVICE}")

Using RUN_DIR: dataset_run_20260912-182558
DEVICE: cpu


In [33]:
# ============================================================
# VERIFY DATASET FILES
# ============================================================

npz_path = DATASET_FILE
splits_path = SPLITS_FILE

assert npz_path.exists(), f"Missing dataset file: {npz_path}"
assert splits_path.exists(), f"Missing splits file: {splits_path}"

print("Found dataset:", npz_path.resolve())
print("Found splits :", splits_path.resolve())

Found dataset: C:\Users\Amar\Downloads\3D tests deeponet battery\dataset_run_20260912-182558\slice_dataset_3d.npz
Found splits : C:\Users\Amar\Downloads\3D tests deeponet battery\dataset_run_20260912-182558\splits.json


In [34]:
# ============================================================
# LOAD UPDATED 3D SEQUENCE SLICE DATASET
# ============================================================

npz = np.load(npz_path, allow_pickle=True)

# Axis-wise STATIC slice inputs and SEQUENCE targets
Xstatic_x = npz["Xstatic_x"].astype(np.float32)   # [C, Sx, Cin, H, W]
Y_x       = npz["Y_x"].astype(np.float32)         # [C, Sx, T, 2, H, W]

Xstatic_y = npz["Xstatic_y"].astype(np.float32)   # [C, Sy, Cin, H, W]
Y_y       = npz["Y_y"].astype(np.float32)         # [C, Sy, T, 2, H, W]

Xstatic_z = npz["Xstatic_z"].astype(np.float32)   # [C, Sz, Cin, H, W]
Y_z       = npz["Y_z"].astype(np.float32)         # [C, Sz, T, 2, H, W]

# Optional full 3D fields
Q_all = npz["Q_all"].astype(np.float32) if "Q_all" in npz.files else None
T_all = npz["T_all"].astype(np.float32) if "T_all" in npz.files else None
f_all = npz["f_all"].astype(np.float32) if "f_all" in npz.files else None
bat_mask_all = npz["bat_mask_all"].astype(np.float32) if "bat_mask_all" in npz.files else None

# Time values and bookkeeping
times_np    = npz["times"].astype(np.float32) if "times" in npz.files else np.array(save_times, dtype=np.float32)
case_ids_np = npz["case_ids"].astype(np.int64) if "case_ids" in npz.files else np.arange(Xstatic_x.shape[0], dtype=np.int64)
meta        = npz["meta"].item() if "meta" in npz.files else {}

# ------------------------------------------------------------
# Basic dimensions
# ------------------------------------------------------------
C_x, Sx, Cin_x, Hx, Wx = Xstatic_x.shape
C_y, Sy, Cin_y, Hy, Wy = Xstatic_y.shape
C_z, Sz, Cin_z, Hz, Wz = Xstatic_z.shape

C_yx, Sx_y, T,   Cout_x, Hx_y, Wx_y = Y_x.shape
C_yy, Sy_y, T_y, Cout_y, Hy_y, Wy_y = Y_y.shape
C_zy, Sz_y, T_z, Cout_z, Hz_y, Wz_y = Y_z.shape

assert C_x == C_y == C_z, "Mismatch in number of cases across Xstatic_x, Xstatic_y, Xstatic_z"
assert C_yx == C_x and C_yy == C_y and C_zy == C_z, "Mismatch between Xstatic_* and Y_* case counts"

assert Sx == Sx_y, "Mismatch between Xstatic_x and Y_x slice counts"
assert Sy == Sy_y, "Mismatch between Xstatic_y and Y_y slice counts"
assert Sz == Sz_y, "Mismatch between Xstatic_z and Y_z slice counts"

assert Hx == Hx_y and Wx == Wx_y, "Mismatch between Xstatic_x and Y_x spatial dimensions"
assert Hy == Hy_y and Wy == Wy_y, "Mismatch between Xstatic_y and Y_y spatial dimensions"
assert Hz == Hz_y and Wz == Wz_y, "Mismatch between Xstatic_z and Y_z spatial dimensions"

assert T == T_y == T_z, "Mismatch in number of time snapshots across axes"
assert Cin_x == Cin_y == Cin_z, "Mismatch in input channel count across axes"
assert Cout_x == Cout_y == Cout_z == 2, "Target channel dimension must be 2: [T, f]"
assert len(times_np) == T, "times_np length must match target sequence length T"

Cin = Cin_x
Cout = Cout_x
NUM_CASES = C_x

# ------------------------------------------------------------
# Window configuration (for multi-slice learning)
# ------------------------------------------------------------

assert WINDOW_SIZE % 2 == 1, "WINDOW_SIZE must be odd"

print("\nWindow config:")
print("WINDOW_SIZE  :", WINDOW_SIZE)
print("WINDOW_RADIUS:", WINDOW_RADIUS)

# ------------------------------------------------------------
# (Optional) Helper for debugging slice windows
# ------------------------------------------------------------
def debug_window_indices(center_idx, n_slices):
    ids = []
    for off in range(-WINDOW_RADIUS, WINDOW_RADIUS + 1):
        j = center_idx + off
        j = max(0, min(n_slices - 1, j))  # clamp
        ids.append(j)
    return ids

# Example debug (first few slices)
print("\nSample window indices (x-axis):")
for i in range(min(3, Sx)):
    print(f"center {i} ->", debug_window_indices(i, Sx))

# ------------------------------------------------------------
# Print summary
# ------------------------------------------------------------
print("\nData summary:")
print("Xstatic_x :", Xstatic_x.shape)
print("Y_x       :", Y_x.shape)
print("Xstatic_y :", Xstatic_y.shape)
print("Y_y       :", Y_y.shape)
print("Xstatic_z :", Xstatic_z.shape)
print("Y_z       :", Y_z.shape)

if Q_all is not None:
    print("Q_all     :", Q_all.shape)
if T_all is not None:
    print("T_all     :", T_all.shape)
if f_all is not None:
    print("f_all     :", f_all.shape)
if bat_mask_all is not None:
    print("bat_mask  :", bat_mask_all.shape)
else:
    print("bat_mask  : not in npz (older dataset run)")

print("times     :", times_np.shape, times_np[:min(5, len(times_np))], "...")
print("case_ids  :", case_ids_np.shape)
print("Cin       :", Cin)
print("Cout      :", Cout)
print("NUM_CASES :", NUM_CASES)
print("meta keys :", list(meta.keys()))


Window config:
WINDOW_SIZE  : 13
WINDOW_RADIUS: 6

Sample window indices (x-axis):
center 0 -> [0, 0, 0, 0, 0, 0, 0, 1, 2, 3, 4, 5, 6]
center 1 -> [0, 0, 0, 0, 0, 0, 1, 2, 3, 4, 5, 6, 7]
center 2 -> [0, 0, 0, 0, 0, 1, 2, 3, 4, 5, 6, 7, 8]

Data summary:
Xstatic_x : (500, 15, 18, 24, 24)
Y_x       : (500, 15, 24, 2, 24, 24)
Xstatic_y : (500, 15, 18, 24, 24)
Y_y       : (500, 15, 24, 2, 24, 24)
Xstatic_z : (500, 15, 18, 24, 24)
Y_z       : (500, 15, 24, 2, 24, 24)
Q_all     : (500, 24, 24, 24)
T_all     : (500, 24, 24, 24, 24)
f_all     : (500, 24, 24, 24, 24)
bat_mask  : (500, 24, 24, 24)
times     : (24,) [100. 200. 300. 400. 500.] ...
case_ids  : (500,)
Cin       : 18
Cout      : 2
NUM_CASES : 500
meta keys : ['format', 'Nx', 'Ny', 'Nz', 'Lx', 'Ly', 'Lz', 'Hx', 'Wx', 'Hy', 'Wy', 'Hz', 'Wz', 'pad_h_x', 'pad_w_x', 'pad_h_y', 'pad_w_y', 'pad_h_z', 'pad_w_z', 'Cin_static', 'Cout', 'times', 'n_slices_x', 'n_slices_y', 'n_slices_z', 'slice_idx_x', 'slice_idx_y', 'slice_idx_z', 'slice_val_x

In [35]:
# ============================================================
# LOAD TRAIN / VAL / TEST SPLITS
# ============================================================

with open(splits_path, "r") as f:
    splits = json.load(f)

train_ids = np.array(splits["train"], dtype=np.int64)
val_ids   = np.array(splits["val"], dtype=np.int64)
test_ids  = np.array(splits["test"], dtype=np.int64)

print("Split sizes:", len(train_ids), len(val_ids), len(test_ids))


Split sizes: 400 50 50


In [36]:
def compute_train_stats_unified(
    Xstatic_x, Y_x,
    Xstatic_y, Y_y,
    Xstatic_z, Y_z,
    train_ids,
    target="f",
    eps=1e-6
):
    """
    Compute normalization stats for SEQUENCE dataset.

    Xstatic_* : [C, S, Cin, H, W]
    Y_*       : [C, S, T, 2, H, W]
    """

    tgt_idx = 0 if target == "T" else 1

    # --------------------------------------------------------
    # Select training cases
    # --------------------------------------------------------
    Xx_tr = Xstatic_x[train_ids]   # [Ctr, Sx, Cin, H, W]
    Xy_tr = Xstatic_y[train_ids]   # [Ctr, Sy, Cin, H, W]
    Xz_tr = Xstatic_z[train_ids]   # [Ctr, Sz, Cin, H, W]

    Yx_tr = Y_x[train_ids, :, :, tgt_idx]   # [Ctr, Sx, T, H, W]
    Yy_tr = Y_y[train_ids, :, :, tgt_idx]   # [Ctr, Sy, T, H, W]
    Yz_tr = Y_z[train_ids, :, :, tgt_idx]   # [Ctr, Sz, T, H, W]

    # --------------------------------------------------------
    # INPUT STATS
    # per-channel mean/std across:
    # (case, slice, height, width)
    # --------------------------------------------------------
    x_mean_x = Xx_tr.mean(axis=(0, 1, 3, 4))   # [Cin]
    x_mean_y = Xy_tr.mean(axis=(0, 1, 3, 4))
    x_mean_z = Xz_tr.mean(axis=(0, 1, 3, 4))

    x_std_x = Xx_tr.std(axis=(0, 1, 3, 4))
    x_std_y = Xy_tr.std(axis=(0, 1, 3, 4))
    x_std_z = Xz_tr.std(axis=(0, 1, 3, 4))

    # average across axes
    x_mean = (x_mean_x + x_mean_y + x_mean_z) / 3.0
    x_std  = (x_std_x  + x_std_y  + x_std_z)  / 3.0
    x_std  = x_std + eps

    # reshape for broadcasting
    x_mean = x_mean[None, :, None, None].astype(np.float32)
    x_std  = x_std[None, :, None, None].astype(np.float32)

    # --------------------------------------------------------
    # OUTPUT STATS
    # across:
    # (case, slice, time, height, width)
    # --------------------------------------------------------
    y_sum = (
        Yx_tr.astype(np.float64).sum() +
        Yy_tr.astype(np.float64).sum() +
        Yz_tr.astype(np.float64).sum()
    )

    y_count = (
        Yx_tr.size +
        Yy_tr.size +
        Yz_tr.size
    )

    y_mean = y_sum / y_count

    y_sq_sum = (
        ((Yx_tr.astype(np.float64) - y_mean) ** 2).sum() +
        ((Yy_tr.astype(np.float64) - y_mean) ** 2).sum() +
        ((Yz_tr.astype(np.float64) - y_mean) ** 2).sum()
    )

    y_std = np.sqrt(y_sq_sum / y_count) + eps

    return {
        "x_mean": x_mean,
        "x_std": x_std,
        "y_mean": float(y_mean),
        "y_std": float(y_std),
    }

# ------------------------------------------------------------
# Compute stats for current target mode
# TARGET = "T" or "f"
# ------------------------------------------------------------
stats = compute_train_stats_unified(
    Xstatic_x, Y_x,
    Xstatic_y, Y_y,
    Xstatic_z, Y_z,
    train_ids=train_ids,
    target=TARGET
)

print("Target mode :", TARGET)
print("Y mean/std  :", stats["y_mean"], stats["y_std"])
print("X mean/std shapes:", stats["x_mean"].shape, stats["x_std"].shape)

Target mode : f
Y mean/std  : 0.36569308105746573 0.45668874375168395
X mean/std shapes: (1, 18, 1, 1) (1, 18, 1, 1)


In [37]:
class Unified3DSliceDataset(Dataset):
    """
    Unified dataset for 3D slice SEQUENCE learning with one common 2D model.

    One sample corresponds to:
        (case_id, axis, slice_id)

    Expected input arrays
    ---------------------
    Xstatic_x, Xstatic_y, Xstatic_z : np.ndarray
        Shape: [C, S, Cin, H, W]

    Y_x, Y_y, Y_z : np.ndarray
        Shape: [C, S, T, 2, H, W]
        target channel 0 -> temperature
        target channel 1 -> liquid fraction

    times : np.ndarray
        Shape: [T]

    ids : array-like
        Case indices to include in this dataset split

    stats : dict
        Must contain:
            "x_mean": [1, Cin, 1, 1]
            "x_std" : [1, Cin, 1, 1]
            "y_mean": scalar
            "y_std" : scalar

    target : str
        "T" for temperature
        "f" for liquid fraction
    """

    AXIS_TO_ID = {"x": 0, "y": 1, "z": 2}
    ID_TO_AXIS = {0: "x", 1: "y", 2: "z"}

    def __init__(
        self,
        Xstatic_x, Y_x,
        Xstatic_y, Y_y,
        Xstatic_z, Y_z,
        times,
        ids,
        stats,
        target="T",
        window_size=3,
        window_mode="clamp"
    ):
        super().__init__()

        assert target in ["T", "f"]
        assert window_size % 2 == 1, "window_size must be odd"

        self.X_by_axis = {
            "x": Xstatic_x,
            "y": Xstatic_y,
            "z": Xstatic_z,
        }
        self.Y_by_axis = {
            "x": Y_x,
            "y": Y_y,
            "z": Y_z,
        }

        self.times = np.asarray(times, dtype=np.float32)
        self.ids = np.asarray(ids, dtype=np.int64)
        self.stats = stats
        self.target = target
        self.target_idx = 0 if target == "T" else 1

        self.t_scale = float(np.max(self.times)) if float(np.max(self.times)) > 0.0 else 1.0

        self.x_mean = self.stats["x_mean"][0].astype(np.float32)
        self.x_std  = self.stats["x_std"][0].astype(np.float32)
        self.y_mean = float(self.stats["y_mean"])
        self.y_std  = float(self.stats["y_std"])

        self.window_size = window_size
        self.window_radius = window_size // 2
        self.window_mode = window_mode

        # ---- Build samples ----
        self.samples = []

        for case_id in self.ids:
            for axis_name in ["x", "y", "z"]:
                X_axis = self.X_by_axis[axis_name]
                _, S_axis, _, _, _ = X_axis.shape

                for slice_idx in range(S_axis):
                    self.samples.append({
                        "case_id": int(case_id),
                        "axis_name": axis_name,
                        "axis_id": self.AXIS_TO_ID[axis_name],
                        "slice_idx": int(slice_idx),
                    })

    def __len__(self):
        return len(self.samples)

    # --------------------------------------------------------
    # NEW: window index helper
    # --------------------------------------------------------
    def _get_window_ids(self, center_idx, n_slices):
        ids = []

        for off in range(-self.window_radius, self.window_radius + 1):
            j = center_idx + off

            if self.window_mode == "clamp":
                j = max(0, min(n_slices - 1, j))

            elif self.window_mode == "reflect":
                if j < 0:
                    j = -j
                if j >= n_slices:
                    j = 2 * n_slices - 2 - j
                j = max(0, min(n_slices - 1, j))

            else:
                raise ValueError("Unknown window_mode")

            ids.append(j)

        return ids

    # --------------------------------------------------------
    # MAIN CHANGE HERE
    # --------------------------------------------------------
    def __getitem__(self, i):

        info = self.samples[i]

        case_id   = info["case_id"]
        axis_name = info["axis_name"]
        axis_id   = info["axis_id"]
        slice_idx = info["slice_idx"]

        X_axis = self.X_by_axis[axis_name]   # [C,S,Cin,H,W]
        Y_axis = self.Y_by_axis[axis_name]   # [C,S,T,2,H,W]

        _, S_axis, Cin, H, W = X_axis.shape

        # --------------------------------------------------------
        # NEW: MULTI-SLICE WINDOW
        # --------------------------------------------------------
        win_ids = self._get_window_ids(slice_idx, S_axis)

        X_list = []
        for j in win_ids:
            Xj = X_axis[case_id, j].astype(np.float32)
            Xj = (Xj - self.x_mean) / self.x_std
            X_list.append(Xj)

        X_window = np.stack(X_list, axis=0).astype(np.float32)   # [Wn,Cin,H,W]

        # --------------------------------------------------------
        # TARGET (center slice only)
        # --------------------------------------------------------
        Y = Y_axis[case_id, slice_idx, :, self.target_idx].astype(np.float32)
        Y_norm = (Y - self.y_mean) / self.y_std

        # --------------------------------------------------------
        # TIME
        # --------------------------------------------------------
        times_raw = self.times.astype(np.float32)
        times_norm = (times_raw / self.t_scale).astype(np.float32)

        return {
            "X_window": torch.from_numpy(X_window),    # [Wn,Cin,H,W]
            "times": torch.from_numpy(times_norm),     # [T]
            "times_raw": torch.from_numpy(times_raw),

            "Y": torch.from_numpy(Y_norm),             # [T,H,W]
            "Y_raw": torch.from_numpy(Y),

            "case_index": int(case_id),
            "axis_id": int(axis_id),
            "axis_name": axis_name,
            "slice_index": int(slice_idx),
            "window_ids": torch.tensor(win_ids, dtype=torch.long),
        }

In [38]:
# ============================================================
# BUILD DATASETS
# ============================================================
ds_train = Unified3DSliceDataset(
    Xstatic_x, Y_x,
    Xstatic_y, Y_y,
    Xstatic_z, Y_z,
    times=times_np,
    ids=train_ids,
    stats=stats,
    target=TARGET,
    window_size=WINDOW_SIZE
)

ds_val = Unified3DSliceDataset(
    Xstatic_x, Y_x,
    Xstatic_y, Y_y,
    Xstatic_z, Y_z,
    times=times_np,
    ids=val_ids,
    stats=stats,
    target=TARGET,
    window_size=WINDOW_SIZE
)

# ============================================================
# BUILD DATALOADERS
# ============================================================

dl_train = DataLoader(
    ds_train,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

dl_val = DataLoader(
    ds_val,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

# ============================================================
# SANITY CHECK
# ============================================================

batch = next(iter(dl_train))

print("Batch keys   :", batch.keys())

# NEW INPUT
print("X_window     :", batch["X_window"].shape)   # [B, Wn, Cin, H, W]

print("times        :", batch["times"].shape)      # [B, T]
print("times_raw    :", batch["times_raw"].shape)  # [B, T]

print("Y            :", batch["Y"].shape)          # [B, T, H, W]
print("Y_raw        :", batch["Y_raw"].shape)      # [B, T, H, W]

print("case_index   :", batch["case_index"].shape if hasattr(batch["case_index"], "shape") else type(batch["case_index"]))
print("axis_id      :", batch["axis_id"].shape if hasattr(batch["axis_id"], "shape") else type(batch["axis_id"]))
print("slice_index  :", batch["slice_index"].shape if hasattr(batch["slice_index"], "shape") else type(batch["slice_index"]))

# NEW DEBUG
print("window_ids   :", batch["window_ids"].shape)  # [B, Wn]

c:\Users\Amar\AppData\Local\Programs\Python\Python314\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Batch keys   : dict_keys(['X_window', 'times', 'times_raw', 'Y', 'Y_raw', 'case_index', 'axis_id', 'axis_name', 'slice_index', 'window_ids'])
X_window     : torch.Size([16, 13, 18, 24, 24])
times        : torch.Size([16, 24])
times_raw    : torch.Size([16, 24])
Y            : torch.Size([16, 24, 24, 24])
Y_raw        : torch.Size([16, 24, 24, 24])
case_index   : torch.Size([16])
axis_id      : torch.Size([16])
slice_index  : torch.Size([16])
window_ids   : torch.Size([16, 13])


In [39]:
# ============================================================
# BASIC ACTIVATION HELPER
# ============================================================

def get_activation(name: str = "relu"):
    name = name.lower()
    if name == "relu":
        return nn.ReLU(inplace=True)
    elif name == "leakyrelu":
        return nn.LeakyReLU(0.2, inplace=True)
    elif name == "gelu":
        return nn.GELU()
    elif name == "silu":
        return nn.SiLU(inplace=True)
    else:
        return nn.ReLU(inplace=True)


# ============================================================
# PAPER-STYLE 2D U-NET BLOCK
# ============================================================

class PaperUNetBlock2D(nn.Module):
    """
    Paper-style U-Net block with fixed internal channel width f.
    Input : [B, f, H, W]
    Output: [B, f, H, W]
    """
    def __init__(self, f: int, dropout: float = 0.0, activation: str = "leakyrelu"):
        super().__init__()

        def conv3(in_ch, out_ch, stride):
            layers = [
                nn.Conv2d(in_ch, out_ch, kernel_size=3, stride=stride, padding=1, bias=False),
                nn.BatchNorm2d(out_ch),
                get_activation(activation),
            ]
            if dropout > 0.0:
                layers.append(nn.Dropout2d(dropout))
            return nn.Sequential(*layers)

        def deconv4(in_ch, out_ch):
            layers = [
                nn.ConvTranspose2d(in_ch, out_ch, kernel_size=4, stride=2, padding=1, bias=True),
                get_activation(activation),
            ]
            if dropout > 0.0:
                layers.append(nn.Dropout2d(dropout))
            return nn.Sequential(*layers)

        self.c1 = conv3(f, f, stride=2)
        self.c2 = conv3(f, f, stride=2)
        self.c3 = conv3(f, f, stride=1)
        self.c4 = conv3(f, f, stride=2)
        self.c5 = conv3(f, f, stride=1)

        self.u1 = deconv4(f,   f)
        self.u2 = deconv4(2*f, f)
        self.u3 = deconv4(2*f, f)

        self.out = nn.Conv2d(2*f, f, kernel_size=3, stride=1, padding=1, bias=True)

    def forward(self, x):
        x_in = x

        s1 = self.c1(x_in)
        s2 = self.c2(s1)
        s3 = self.c3(s2)

        b = self.c4(s3)
        b = self.c5(b)

        x = self.u1(b)
        x = torch.cat([x, s3], dim=1)

        x = self.u2(x)
        x = torch.cat([x, s1], dim=1)

        x = self.u3(x)
        x = torch.cat([x, x_in], dim=1)

        x = self.out(x)
        return x


# ============================================================
# TIME FOURIER FEATURES
# ============================================================

def time_fourier_features(t: torch.Tensor, n_freq: int = 8):
    """
    t: [B, T] in normalized range [0, 1]
    returns: [B, T, 1 + 2*n_freq]
    """
    feats = [t.unsqueeze(-1)]
    for k in range(n_freq):
        w = (2.0 ** k) * torch.pi
        feats.append(torch.sin(w * t).unsqueeze(-1))
        feats.append(torch.cos(w * t).unsqueeze(-1))
    return torch.cat(feats, dim=-1)


# ============================================================
# UNIFIED 2D SLICE SEQUENCE MODEL
# ============================================================

class UnifiedSliceUNet(nn.Module):

    def __init__(
        self,
        in_channels: int = 17,
        window_size: int = 3,
        base_width: int = 64,
        time_dim: int = 64,
        time_n_freq: int = 8,
        dropout: float = 0.0,
        activation: str = "leakyrelu"
    ):
        super().__init__()

        self.in_channels = in_channels
        self.window_size = window_size
        self.base_width = base_width
        self.time_dim = time_dim
        self.time_n_freq = time_n_freq

        fourier_in_dim = 1 + 2 * time_n_freq

        # SAME encoder as before
        self.lift = nn.Conv2d(in_channels, base_width, kernel_size=1, bias=True)

        self.unet1 = PaperUNetBlock2D(base_width, dropout, activation)
        self.unet2 = PaperUNetBlock2D(base_width, dropout, activation)
        self.unet3 = PaperUNetBlock2D(base_width, dropout, activation)

        self.act_mid = get_activation(activation)

        # NEW: slice fusion layer
        self.slice_fuse = nn.Sequential(
            nn.Conv2d(base_width * window_size, base_width, kernel_size=1, bias=False),
            nn.BatchNorm2d(base_width),
            get_activation(activation),
        )

        # time branch (UNCHANGED)
        self.time_mlp = nn.Sequential(
            nn.Linear(fourier_in_dim, time_dim),
            get_activation(activation),
            nn.Linear(time_dim, time_dim),
            get_activation(activation),
        )

        self.fuse = nn.Sequential(
            nn.Conv2d(base_width + time_dim, base_width, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(base_width),
            get_activation(activation),

            nn.Conv2d(base_width, base_width, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(base_width),
            get_activation(activation),

            nn.Conv2d(base_width, 1, kernel_size=1, bias=True)
        )

    # --------------------------------------------------------
    # Encode ONE slice (unchanged logic)
    # --------------------------------------------------------
    def encode_one_slice(self, X_slice):
        x = self.lift(X_slice)
        x = self.unet1(x)
        x = self.act_mid(x)
        x = self.unet2(x)
        x = self.act_mid(x)
        x = self.unet3(x)
        return x   # [B, f, H, W]

    # --------------------------------------------------------
    # NEW: Encode WINDOW of slices
    # --------------------------------------------------------
    def encode_window(self, X_window):
        # X_window: [B, Wn, Cin, H, W]

        B, Wn, Cin, H, W = X_window.shape

        feats = []
        for w in range(Wn):
            f_w = self.encode_one_slice(X_window[:, w])   # [B,f,H,W]
            feats.append(f_w)

        feat_cat = torch.cat(feats, dim=1)   # [B, Wn*f, H, W]
        feat = self.slice_fuse(feat_cat)     # [B, f, H, W]

        return feat

    # --------------------------------------------------------
    # FORWARD (UPDATED)
    # --------------------------------------------------------
    def forward(self, X_window, times):

        if times.dim() == 1:
            times = times.unsqueeze(0).expand(X_window.shape[0], -1)

        B, Wn, Cin, H, W = X_window.shape
        Tn = times.shape[1]

        # NEW: multi-slice encoding
        feat = self.encode_window(X_window)   # [B, f, H, W]

        # expand spatial encoding across time
        feat_bt = feat.unsqueeze(1).expand(B, Tn, self.base_width, H, W)

        # time embedding
        t_feat = time_fourier_features(times, n_freq=self.time_n_freq)
        t_emb = self.time_mlp(t_feat)
        t_emb = t_emb.unsqueeze(-1).unsqueeze(-1).expand(B, Tn, self.time_dim, H, W)

        # fuse spatial + time
        z = torch.cat([feat_bt, t_emb], dim=2)
        z = z.reshape(B * Tn, self.base_width + self.time_dim, H, W)

        y = self.fuse(z)
        y = y.reshape(B, Tn, H, W)

        return y

In [40]:
# ============================================================
# MODEL + OPTIMIZER + LOSS / METRICS
# ============================================================

model = UnifiedSliceUNet(
    in_channels=IN_CHANNELS,      # still 17 (per slice)
    window_size=WINDOW_SIZE,      # NEW
    base_width=UNET_FEATURES,
    time_dim=TIME_DIM,
    time_n_freq=TIME_N_FREQ,
    dropout=UNET_DROPOUT,
    activation=ACTIVATION_FUNCTION
).to(DEVICE)

opt = torch.optim.Adam(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

# ------------------------------------------------------------
# Loss
# ------------------------------------------------------------
def mse_loss(pred, target):
    return torch.mean((pred - target) ** 2)


def temporal_smoothness_loss(pred):
    if pred.shape[1] < 2:
        return pred.new_tensor(0.0)
    return torch.mean((pred[:, 1:] - pred[:, :-1]) ** 2)


# ------------------------------------------------------------
# Metrics
# ------------------------------------------------------------
def rmse(pred, target):
    return torch.sqrt(torch.mean((pred - target) ** 2) + 1e-12)


def denorm_field(x, y_mean, y_std):
    return x * y_std + y_mean


def rmse_physical(pred, target, y_mean, y_std):
    pred_phys = denorm_field(pred, y_mean, y_std)
    tgt_phys  = denorm_field(target, y_mean, y_std)
    return torch.sqrt(torch.mean((pred_phys - tgt_phys) ** 2) + 1e-12)


print(model)


# ============================================================
# UPDATED RUN EPOCH
# ============================================================

def run_epoch(model, loader, train: bool, smoothness_weight: float = 0.0):

    model.train(train)

    total_loss = 0.0
    total_rmse = 0.0
    total_rmse_phys = 0.0
    n = 0

    for batch in loader:

        # ----------------------------
        # NEW INPUT
        # ----------------------------
        X_window = batch["X_window"].to(DEVICE, non_blocking=True)   # [B,Wn,Cin,H,W]
        times    = batch["times"].to(DEVICE, non_blocking=True)      # [B,T]
        Y        = batch["Y"].to(DEVICE, non_blocking=True)          # [B,T,H,W]

        if train:
            opt.zero_grad(set_to_none=True)

        # ----------------------------
        # MODEL FORWARD
        # ----------------------------
        pred = model(X_window, times)                                # [B,T,H,W]

        # ----------------------------
        # LOSS
        # ----------------------------
        loss_main = mse_loss(pred, Y)
        loss_smooth = temporal_smoothness_loss(pred)
        loss = loss_main + smoothness_weight * loss_smooth

        if train:
            loss.backward()
            opt.step()

        # ----------------------------
        # METRICS
        # ----------------------------
        with torch.no_grad():
            r = rmse(pred, Y)
            r_phys = rmse_physical(
                pred, Y,
                y_mean=stats["y_mean"],
                y_std=stats["y_std"]
            )

        bs = X_window.shape[0]   # changed
        total_loss += float(loss.detach()) * bs
        total_rmse += float(r.detach()) * bs
        total_rmse_phys += float(r_phys.detach()) * bs
        n += bs

    return {
        "loss": total_loss / max(n, 1),
        "rmse": total_rmse / max(n, 1),
        "rmse_physical": total_rmse_phys / max(n, 1),
    }

UnifiedSliceUNet(
  (lift): Conv2d(18, 64, kernel_size=(1, 1), stride=(1, 1))
  (unet1): PaperUNetBlock2D(
    (c1): Sequential(
      (0): Conv2d(64, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
    )
    (c2): Sequential(
      (0): Conv2d(64, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
    )
    (c3): Sequential(
      (0): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
    )
    (c4): Sequential(
      (0): Conv2d(64, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, tra

In [41]:
# ============================================================
# OPTIONAL TORCH / OPTIMIZER SANITY CHECK
# ============================================================
print("Torch version:", torch.__version__)
importlib.import_module("torch._utils")

_ = optim.Adam(
    [torch.nn.Parameter(torch.randn(2, requires_grad=True))],
    lr=1e-3
)
print("Adam OK")

Torch version: 2.11.0+cpu
Adam OK


In [42]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device used: {device}")
print("\n")


Device used: cpu




In [ ]:
# ============================================================
# CHECKPOINT DIRECTORY
# ============================================================

ckpt_dir = RUN_DIR / f"checkpoints_unified_slice_unet_{TARGET}"
ckpt_dir.mkdir(parents=True, exist_ok=True)

best_val = float("inf")
history = {
    "epoch": [],
    "train_loss": [],
    "train_rmse": [],
    "train_rmse_physical": [],
    "val_loss": [],
    "val_rmse": [],
    "val_rmse_physical": [],
}

# ============================================================
# TRAINING LOOP
# ============================================================

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()

    train_metrics = run_epoch(
        model,
        dl_train,
        train=True
    )
    val_metrics = run_epoch(
        model,
        dl_val,
        train=False
    )

    dt = time.time() - t0

    tr_loss = train_metrics["loss"]
    tr_rmse = train_metrics["rmse"]
    tr_rmse_phys = train_metrics["rmse_physical"]

    va_loss = val_metrics["loss"]
    va_rmse = val_metrics["rmse"]
    va_rmse_phys = val_metrics["rmse_physical"]

    history["epoch"].append(epoch)
    history["train_loss"].append(tr_loss)
    history["train_rmse"].append(tr_rmse)
    history["train_rmse_physical"].append(tr_rmse_phys)
    history["val_loss"].append(va_loss)
    history["val_rmse"].append(va_rmse)
    history["val_rmse_physical"].append(va_rmse_phys)

    print(
        f"epoch {epoch:03d} | "
        f"train loss {tr_loss:.4e} rmse {tr_rmse:.4e} rmse_phys {tr_rmse_phys:.4e} | "
        f"val loss {va_loss:.4e} rmse {va_rmse:.4e} rmse_phys {va_rmse_phys:.4e} | "
        f"{dt:.1f}s"
    )

    # --------------------------------------------------------
    # Save best checkpoint using validation loss
    # --------------------------------------------------------
    if va_loss < best_val:
        best_val = va_loss
        ckpt_path = ckpt_dir / "best.pt"

        torch.save(
            {
                "model": model.state_dict(),
                "opt": opt.state_dict(),
                "epoch": epoch,
                "best_val": best_val,

                # dataset / normalization info
                "stats": stats,
                "meta": meta,
                "times": times_np,
                "target": TARGET,

                # model config
                "in_channels": IN_CHANNELS,
                "window_size": WINDOW_SIZE,
                "base_width": UNET_FEATURES,
                "time_dim": TIME_DIM,
                "time_n_freq": TIME_N_FREQ,
                "dropout": UNET_DROPOUT,
                "activation": ACTIVATION_FUNCTION,

                # training / loss config
                "smoothness_weight": SMOOTHNESS_WEIGHT,
                "grad_weight": GRAD_WEIGHT,
                "interface_weight": INTERFACE_WEIGHT,
                "interface_center": INTERFACE_CENTER,
                "interface_sigma": INTERFACE_SIGMA,
                "interface_alpha": INTERFACE_ALPHA,

                # optional
                "history": history,
            },
            ckpt_path
        )
        print("  saved:", ckpt_path)

# ============================================================
# SAVE FINAL CHECKPOINT + TRAINING HISTORY
# ============================================================

final_ckpt_path = ckpt_dir / "last.pt"
torch.save(
    {
        "model": model.state_dict(),
        "opt": opt.state_dict(),
        "epoch": EPOCHS,
        "best_val": best_val,

        "stats": stats,
        "meta": meta,
        "times": times_np,
        "target": TARGET,

        "in_channels": IN_CHANNELS,
        "window_size": WINDOW_SIZE,
        "base_width": UNET_FEATURES,
        "time_dim": TIME_DIM,
        "time_n_freq": TIME_N_FREQ,
        "dropout": UNET_DROPOUT,
        "activation": ACTIVATION_FUNCTION,

        "smoothness_weight": SMOOTHNESS_WEIGHT,
        "grad_weight": GRAD_WEIGHT,
        "interface_weight": INTERFACE_WEIGHT,
        "interface_center": INTERFACE_CENTER,
        "interface_sigma": INTERFACE_SIGMA,
        "interface_alpha": INTERFACE_ALPHA,

        "history": history,
    },
    final_ckpt_path
)

print("Final checkpoint saved:", final_ckpt_path)

epoch 001 | train loss 2.8557e-01 rmse 5.1696e-01 rmse_phys 2.3609e-01 | val loss 1.9478e-01 rmse 4.3241e-01 rmse_phys 1.9747e-01 | 4606.8s
  saved: dataset_run_20260912-182558\checkpoints_unified_slice_unet_f\best.pt
epoch 002 | train loss 1.2612e-01 rmse 3.5191e-01 rmse_phys 1.6071e-01 | val loss 1.0595e-01 rmse 3.1834e-01 rmse_phys 1.4538e-01 | 4294.3s
  saved: dataset_run_20260912-182558\checkpoints_unified_slice_unet_f\best.pt
epoch 003 | train loss 9.8189e-02 rmse 3.1002e-01 rmse_phys 1.4158e-01 | val loss 1.2742e-01 rmse 3.5090e-01 rmse_phys 1.6025e-01 | 4659.4s
epoch 004 | train loss 8.3274e-02 rmse 2.8580e-01 rmse_phys 1.3052e-01 | val loss 9.9137e-02 rmse 3.0837e-01 rmse_phys 1.4083e-01 | 4749.0s
  saved: dataset_run_20260912-182558\checkpoints_unified_slice_unet_f\best.pt
epoch 005 | train loss 7.7133e-02 rmse 2.7426e-01 rmse_phys 1.2525e-01 | val loss 6.3409e-02 rmse 2.4264e-01 rmse_phys 1.1081e-01 | 5963.4s
  saved: dataset_run_20260912-182558\checkpoints_unified_slice_une

In [43]:
def get_axis_pad(meta, axis):
    pad_h = int(meta.get(f"pad_h_{axis}", 0))
    pad_w = int(meta.get(f"pad_w_{axis}", 0))
    return pad_h, pad_w


def unpad_2d(arr, pad_h=0, pad_w=0):
    
    H, W = arr.shape
    h_end = H - pad_h if pad_h > 0 else H
    w_end = W - pad_w if pad_w > 0 else W
    return arr[:h_end, :w_end]

In [44]:
# ============================================================
# UNCOMPRESSED .npy CACHE  (one-off cost, then memory-mapped loads)
#
# slice_dataset_3d.npz is ~4.8 GB of zlib-compressed float32. np.load has to
# inflate each array in full before you can touch a single slice, which costs
# ~4 minutes of every session. Writing the arrays the inference path needs back
# out as plain .npy lets numpy memory-map them: the load is milliseconds and
# only the pages you actually index are read from disk.
#
# Price: ~3.5 GB of disk next to the npz, written once. Delete the *_npy folder
# to reclaim it; it is rebuilt automatically if the npz changes.
#
# T_all / f_all are deliberately NOT cached - the inference path never reads
# them, and they are another 1.3 GB.
# ============================================================

NPY_CACHE_ARRAYS = (
    "Xstatic_x", "Y_x",
    "Xstatic_y", "Y_y",
    "Xstatic_z", "Y_z",
    "Q_all", "bat_mask_all",
    "times", "case_ids",
)


def npy_cache_dir(npz_path):
    p = Path(npz_path)
    return p.parent / (p.stem + "_npy")


def _cache_manifest(npz_path):
    st = Path(npz_path).stat()
    return {"npz_size": st.st_size, "npz_mtime": int(st.st_mtime)}


def build_npy_cache(npz_path, names=NPY_CACHE_ARRAYS, force=False, verbose=True):
    """
    Decompress the needed arrays into <dataset>_npy/*.npy exactly once.

    Returns the cache directory. Safe to call every session - it is a no-op once
    the manifest matches the npz.
    """
    cdir = npy_cache_dir(npz_path)
    man_p = cdir / "manifest.json"
    want = _cache_manifest(npz_path)

    if not force and man_p.exists():
        try:
            have = json.loads(man_p.read_text(encoding="utf-8"))
            if (have.get("npz") == want
                    and all((cdir / f"{n}.npy").exists() for n in have.get("names", []))
                    and (cdir / "meta.npy").exists()):
                if verbose:
                    print(f"  npy cache up to date: {cdir.name}")
                return cdir
        except Exception:
            pass

    cdir.mkdir(exist_ok=True)
    if verbose:
        print(f"  building npy cache in {cdir.name} (one-off, a few minutes)...")

    npz = np.load(npz_path, allow_pickle=True)
    written = []
    for n in names:
        if n not in npz.files:
            if verbose:
                print(f"    skip {n} (not in npz)")
            continue
        t0 = time.perf_counter()
        arr = npz[n]                                   # inflate once
        np.save(cdir / f"{n}.npy", arr)
        written.append(n)
        if verbose:
            print(f"    {n:14s} {str(arr.shape):28s} "
                  f"{arr.nbytes / 1e6:8.1f} MB  {time.perf_counter() - t0:5.1f}s")
        del arr

    # meta is an object array, so it cannot be memory-mapped - pickle it as is
    meta = npz["meta"].item() if "meta" in npz.files else {}
    np.save(cdir / "meta.npy", np.array(meta, dtype=object), allow_pickle=True)

    man_p.write_text(json.dumps({"npz": want, "names": written}), encoding="utf-8")
    if verbose:
        print(f"  npy cache built: {len(written)} arrays")
    return cdir


def load_arrays_mmap(npz_path, names=NPY_CACHE_ARRAYS, verbose=True):
    """
    Memory-mapped view of the cached arrays, plus meta.

    Falls back to reading the npz directly if the cache cannot be built (for
    example a read-only dataset folder).
    """
    try:
        cdir = build_npy_cache(npz_path, names=names, verbose=verbose)
    except Exception as exc:
        print(f"  npy cache unavailable ({exc}) - falling back to the npz")
        npz = np.load(npz_path, allow_pickle=True)
        out = {n: npz[n] for n in names if n in npz.files}
        out["meta"] = npz["meta"].item() if "meta" in npz.files else {}
        return out

    t0 = time.perf_counter()
    out = {}
    for n in names:
        p = cdir / f"{n}.npy"
        if p.exists():
            out[n] = np.load(p, mmap_mode="r")
    out["meta"] = np.load(cdir / "meta.npy", allow_pickle=True).item()
    if verbose:
        print(f"  mmapped {len(out) - 1} arrays in {1e3 * (time.perf_counter() - t0):.0f} ms")
    return out


In [45]:
# ============================================================
# CACHED LOADER : checkpoint + npz are read ONCE per session
#
# The previous driver called infer_and_plot_from_ckpt() inside a loop, so the
# whole npz (hundreds of MB) and the checkpoint were re-read for every single
# slice plot. That also makes any timing measurement meaningless. Load once,
# reuse the bundle.
# ============================================================

_SESSION_CACHE = {}


def load_model_and_dataset(ckpt_path, npz_path, split="test",
                          force_reload=False, use_npy_cache=True):
    """
    Returns a bundle dict:
        model, ds, stats, meta, times, target, split_ids,
        Q_all, bat_mask_all, window_size
    """
    key = (str(ckpt_path), str(npz_path), str(split), bool(use_npy_cache))
    if (not force_reload) and (key in _SESSION_CACHE):
        return _SESSION_CACHE[key]

    ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)

    stats  = ckpt["stats"]
    times  = np.asarray(ckpt["times"], dtype=np.float32)
    target = ckpt["target"]
    window_size = int(ckpt.get("window_size", 3))

    # Memory-mapped .npy cache instead of inflating ~3.4 GB of zlib every time.
    # .astype(np.float32) is deliberately NOT called here: these arrays are
    # already float32, and the copy would materialise the whole thing in RAM and
    # defeat the memory map. _as_f32 only converts if it actually has to.
    if use_npy_cache:
        arrs = load_arrays_mmap(npz_path)
    else:
        _npz = np.load(npz_path, allow_pickle=True)
        arrs = {n: _npz[n] for n in NPY_CACHE_ARRAYS if n in _npz.files}
        arrs["meta"] = _npz["meta"].item() if "meta" in _npz.files else {}

    def _as_f32(a):
        if a is None:
            return None
        return a if a.dtype == np.float32 else np.asarray(a, dtype=np.float32)

    Xstatic_x = _as_f32(arrs["Xstatic_x"])
    Y_x       = _as_f32(arrs["Y_x"])
    Xstatic_y = _as_f32(arrs["Xstatic_y"])
    Y_y       = _as_f32(arrs["Y_y"])
    Xstatic_z = _as_f32(arrs["Xstatic_z"])
    Y_z       = _as_f32(arrs["Y_z"])

    meta = arrs.get("meta", {})

    Q_all        = _as_f32(arrs.get("Q_all"))
    bat_mask_all = _as_f32(arrs.get("bat_mask_all"))

    splits_path = Path(npz_path).parent / "splits.json"
    with open(splits_path, "r", encoding="utf-8") as fh:
        splits = json.load(fh)
    split_ids = np.array(splits[split], dtype=np.int64)

    ds = Unified3DSliceDataset(
        Xstatic_x, Y_x,
        Xstatic_y, Y_y,
        Xstatic_z, Y_z,
        times=times,
        ids=split_ids,
        stats=stats,
        target=target,
        window_size=window_size,
    )
    ds.times_raw = times

    model = UnifiedSliceUNet(
        in_channels=ckpt["in_channels"],
        window_size=window_size,
        base_width=ckpt["base_width"],
        time_dim=ckpt.get("time_dim", 64),
        time_n_freq=ckpt.get("time_n_freq", 8),
        dropout=ckpt.get("dropout", 0.0),
        activation=ckpt.get("activation", "leakyrelu"),
    ).to(DEVICE)

    model.load_state_dict(ckpt["model"])
    model.eval()

    bundle = {
        "model": model,
        "ds": ds,
        "stats": stats,
        "meta": meta,
        "times": times,
        "target": target,
        "split_ids": split_ids,
        "Q_all": Q_all,
        "bat_mask_all": bat_mask_all,
        "window_size": window_size,
        "n_params": int(sum(p.numel() for p in model.parameters())),
    }

    _SESSION_CACHE[key] = bundle

    print(f"loaded  | split={split}  cases={len(split_ids)}  "
          f"samples={len(ds)}  window={window_size}  "
          f"params={bundle['n_params']:,}  device={DEVICE}")
    if bat_mask_all is None:
        print("        | no 'bat_mask_all' in npz -> falling back to the Q plateau")

    return bundle


# ------------------------------------------------------------
# WHICH PLANES ACTUALLY CUT THE PRISM
# ------------------------------------------------------------
def get_battery_mask_3d(bundle, case_id, tol=1e-6):
    """
    Battery voxels for one case.

    Prefers the stored mask. Falls back to the Q plateau: the prism is the only
    region with a uniform maximum, and the generator clips the Gaussian
    background well below the battery level, so Q >= Q.max() - tol is exact.
    """
    if bundle["bat_mask_all"] is not None:
        return bundle["bat_mask_all"][int(case_id)] > 0.5

    if bundle["Q_all"] is None:
        raise ValueError("npz has neither 'bat_mask_all' nor 'Q_all'")

    Q = bundle["Q_all"][int(case_id)]
    return Q >= (float(Q.max()) - tol * max(abs(float(Q.max())), 1.0))


def cutting_slice_positions(bundle, case_id, axis):
    """
    Stored slice POSITIONS (0 .. S-1) whose plane intersects the battery.

    meta['slice_idx_*'] holds GRID indices; the dataset and the plotting
    functions index by position. The two are different numbers, which is the
    usual source of confusion here.
    """
    bat  = get_battery_mask_3d(bundle, case_id)
    grid = np.asarray(bundle["meta"][f"slice_idx_{axis}"])
    if grid.ndim == 2:                      # per-case slice indices
        grid = grid[int(case_id)]
    a = "xyz".index(axis)
    return [int(p) for p, g in enumerate(grid) if bat.take(int(g), axis=a).any()]


def slice_coverage_report(bundle, case_id):
    """Print, per axis, which stored planes cut the battery and which do not."""
    print(f"case {case_id}: planes that cut the battery")
    out = {}
    for axis in ["x", "y", "z"]:
        grid = np.asarray(bundle["meta"][f"slice_idx_{axis}"])
        if grid.ndim == 2:
            grid = grid[int(case_id)]
        pos = cutting_slice_positions(bundle, case_id, axis)
        out[axis] = pos
        print(f"  axis {axis}: {len(pos):2d}/{len(grid)}  "
              f"positions {pos}  (grid planes {[int(grid[p]) for p in pos]})")
    return out


In [ ]:
# ============================================================
# INFERENCE TIMING
#
# Two numbers are reported:
#   per case   - every stored slice of all 3 axes in one batched sweep. This is
#                what you pay to reconstruct the full 3D field for one new
#                geometry, and it is the number to quote against a CFD run.
#   per slice  - two flavours, because they answer different questions:
#                  amortised = per-case / n_slices   (throughput)
#                  latency   = batch-size-1 forward  (single-query cost)
#
# One slice prediction covers the FULL time sequence in a single forward pass,
# so the cost per (slice, snapshot) is per-slice / T.
# ============================================================

def r2_score_np(true, pred):
    """
    Coefficient of determination over every element of the two arrays.

    R2 = 1 - SSE/SST with SST taken about the mean of `true`, so R2 = 0 means
    "no better than predicting the mean field" and negative means worse.
    """
    t = np.asarray(true, dtype=np.float64)
    p = np.asarray(pred, dtype=np.float64)
    sse = float(((t - p) ** 2).sum())
    sst = float(((t - t.mean()) ** 2).sum())
    return 1.0 - sse / sst if sst > 0.0 else float("nan")


def _r2_parts(true, pred):
    """
    Sufficient statistics for pooling R2 across axes without keeping the fields.

    SST for the pooled set needs the GLOBAL mean, so store sum(y) and sum(y^2)
    rather than a per-axis SST -- averaging per-axis R2 values is not the same
    number and is wrong whenever the axes have different means.
    """
    t = np.asarray(true, dtype=np.float64)
    p = np.asarray(pred, dtype=np.float64)
    return {
        "n": int(t.size),
        "sum_y": float(t.sum()),
        "sum_y2": float((t ** 2).sum()),
        "sse": float(((t - p) ** 2).sum()),
    }


def _pool_r2(parts):
    """Pooled RMSE and R2 from a list of _r2_parts() dicts."""
    n = sum(q["n"] for q in parts)
    sse = sum(q["sse"] for q in parts)
    sy = sum(q["sum_y"] for q in parts)
    sy2 = sum(q["sum_y2"] for q in parts)
    sst = sy2 - sy * sy / n
    return (float(np.sqrt(sse / n)),
            1.0 - sse / sst if sst > 0.0 else float("nan"))


def _sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()


@torch.no_grad()
def benchmark_case_inference(
    bundle,
    case_id,
    batch_size=16,
    n_warmup=2,
    n_repeat=3,
    n_latency_samples=8,
    include_data_prep=True,
    verbose=True,
):
    """
    Time the trained model on ONE case (all axes, all stored slices).

    Returns a dict of timings in seconds unless the key says ms.
    """
    model = bundle["model"]
    ds    = bundle["ds"]
    model.eval()

    idxs = [i for i, s in enumerate(ds.samples) if int(s["case_id"]) == int(case_id)]
    if not idxs:
        raise ValueError(f"case_id={case_id} is not in this split")

    n_slices = len(idxs)

    # ---------------- host-side sample assembly ----------------
    _t0 = time.perf_counter()
    Xs, Ts = [], []
    for i in idxs:
        s = ds[i]
        Xs.append(s["X_window"])
        Ts.append(s["times"])
    X_cpu = torch.stack(Xs, dim=0)
    T_cpu = torch.stack(Ts, dim=0)
    t_prep = time.perf_counter() - _t0

    n_times = int(T_cpu.shape[1])

    X = X_cpu.to(DEVICE)
    T = T_cpu.to(DEVICE)

    # ---------------- warm-up (cuDNN autotune, lazy init) ----------------
    for _ in range(n_warmup):
        _ = model(X[:min(batch_size, n_slices)], T[:min(batch_size, n_slices)])
    _sync()

    # ---------------- batched sweep over the whole case ----------------
    fwd_times = []
    for _ in range(n_repeat):
        _sync()
        _t1 = time.perf_counter()
        for b in range(0, n_slices, batch_size):
            _ = model(X[b:b + batch_size], T[b:b + batch_size])
        _sync()
        fwd_times.append(time.perf_counter() - _t1)

    t_fwd = float(np.median(fwd_times))

    # ---------------- single-sample latency ----------------
    lat = []
    for i in range(min(n_latency_samples, n_slices)):
        _sync()
        _t2 = time.perf_counter()
        _ = model(X[i:i + 1], T[i:i + 1])
        _sync()
        lat.append(time.perf_counter() - _t2)
    lat = np.asarray(lat, dtype=np.float64)

    t_case_total = t_fwd + (t_prep if include_data_prep else 0.0)

    res = {
        "case_id": int(case_id),
        "n_slices": int(n_slices),
        "n_times": n_times,
        "batch_size": int(batch_size),
        "device": str(DEVICE),

        "t_case_forward_s": t_fwd,
        "t_case_dataprep_s": t_prep,
        "t_case_total_s": t_case_total,

        "t_per_slice_amortised_ms": 1e3 * t_fwd / n_slices,
        "t_per_slice_total_ms": 1e3 * t_case_total / n_slices,
        "t_per_slice_latency_mean_ms": 1e3 * float(lat.mean()),
        "t_per_slice_latency_median_ms": 1e3 * float(np.median(lat)),
        "t_per_slice_latency_p95_ms": 1e3 * float(np.percentile(lat, 95)),

        "t_per_slice_per_snapshot_ms": 1e3 * t_fwd / (n_slices * n_times),
        "slices_per_s": n_slices / t_fwd if t_fwd > 0 else float("nan"),
        "fields_per_s": (n_slices * n_times) / t_fwd if t_fwd > 0 else float("nan"),
    }

    if verbose:
        print(f"case {case_id}  |  {n_slices} slices x {n_times} snapshots  "
              f"|  batch {batch_size}  |  {DEVICE}")
        print(f"  per case : forward {1e3 * t_fwd:8.1f} ms"
              f"   + data prep {1e3 * t_prep:8.1f} ms"
              f"   = {1e3 * t_case_total:8.1f} ms")
        print(f"  per slice: amortised {res['t_per_slice_amortised_ms']:7.2f} ms"
              f"   | latency (b=1) {res['t_per_slice_latency_median_ms']:7.2f} ms"
              f"   | per snapshot {res['t_per_slice_per_snapshot_ms']:6.3f} ms")
        print(f"  throughput: {res['slices_per_s']:.1f} slices/s"
              f"  = {res['fields_per_s']:.0f} 2D fields/s")

    return res


@torch.no_grad()
def benchmark_cases(bundle, case_ids=None, n_cases=5, batch_size=16, **kw):
    """
    Repeat benchmark_case_inference over several cases and summarise.
    Returns a pandas DataFrame (one row per case) and prints mean +/- std.
    """
    if case_ids is None:
        case_ids = bundle["split_ids"][:n_cases]

    rows = [benchmark_case_inference(bundle, int(c), batch_size=batch_size,
                                     verbose=False, **kw) for c in case_ids]
    df = pd.DataFrame(rows)

    cols = [
        "t_case_total_s",
        "t_case_forward_s",
        "t_case_dataprep_s",
        "t_per_slice_amortised_ms",
        "t_per_slice_latency_median_ms",
        "t_per_slice_per_snapshot_ms",
    ]

    print(f"\nInference timing over {len(df)} cases "
          f"({int(df['n_slices'].iloc[0])} slices x {int(df['n_times'].iloc[0])} snapshots each)")
    print(f"device={DEVICE}  batch={batch_size}  "
          f"window={bundle['window_size']}  params={bundle['n_params']:,}")
    for c in cols:
        print(f"  {c:32s} {df[c].mean():10.3f}  +/- {df[c].std():.3f}")

    return df


@torch.no_grad()
def predict_case_fields(bundle, case_id, batch_size=16):
    """
    Predict every stored slice of one case and return the de-normalised fields
    plus the timing, so the numbers you plot and the numbers you time come from
    exactly the same call.

    Returns {axis: {"pred": [S,T,H,W], "true": [S,T,H,W], "positions": [...]}},
    and a timing dict.
    """
    model = bundle["model"]
    ds    = bundle["ds"]
    stats = bundle["stats"]
    model.eval()

    out = {}
    _sync()
    t0 = time.perf_counter()

    for axis in ["x", "y", "z"]:
        idxs = [i for i, s in enumerate(ds.samples)
                if int(s["case_id"]) == int(case_id) and s["axis_name"] == axis]
        if not idxs:
            continue

        Xs = torch.stack([ds[i]["X_window"] for i in idxs]).to(DEVICE)
        Ts = torch.stack([ds[i]["times"] for i in idxs]).to(DEVICE)
        Yt = np.stack([ds[i]["Y_raw"].cpu().numpy() for i in idxs], axis=0)

        preds = []
        for b in range(0, len(idxs), batch_size):
            p = model(Xs[b:b + batch_size], Ts[b:b + batch_size]).cpu().numpy()
            preds.append(p)
        pred = np.concatenate(preds, axis=0) * stats["y_std"] + stats["y_mean"]

        out[axis] = {
            "pred": pred,
            "true": Yt,
            "positions": [int(ds.samples[i]["slice_idx"]) for i in idxs],
            "rmse": float(np.sqrt(np.mean((pred - Yt) ** 2))),
            "r2": r2_score_np(Yt, pred),
        }

    _sync()
    t_total = time.perf_counter() - t0

    n_slices = sum(v["pred"].shape[0] for v in out.values())
    timing = {
        "t_case_total_s": t_total,
        "n_slices": n_slices,
        "t_per_slice_ms": 1e3 * t_total / max(n_slices, 1),
    }

    print(f"case {case_id}: predicted {n_slices} slices in {1e3 * t_total:.1f} ms "
          f"({timing['t_per_slice_ms']:.2f} ms/slice)")
    for axis, v in out.items():
        print(f"  axis {axis}: RMSE({bundle['target']}) = {v['rmse']:.5f}"
              f"   R2 = {v['r2']:.5f}")

    return out, timing


In [ ]:
# ============================================================
# FAST INFERENCE ENGINE
#
# The reference forward() is correct, but it repeats a lot of work when you
# sweep a whole case. Three things dominate on CPU:
#
#   1. encode_one_slice() is called once per (centre slice, window position).
#      With window=13 over 15 stored slices that is 15*13 = 195 encoder passes
#      per axis, while only 15 DISTINCT slices exist -> a 13x overcount. The
#      encoder is per-slice and slice_fuse is a 1x1 conv over the concatenation,
#      so encoding each stored slice once and gathering the window in FEATURE
#      space is exactly the same arithmetic.
#
#   2. fuse[0] is a 3x3 conv over cat([feat, t_emb]) evaluated B*T times, but
#      t_emb is spatially CONSTANT. Split the kernel into its feat half and its
#      time half: the feat half then depends only on the slice (B passes instead
#      of B*T), and a zero-padded 3x3 conv of a constant map takes only 9
#      distinct values over the image (corner / edge / interior), so the time
#      half collapses to a 9-region matmul. Everything after fuse[0] is
#      nonlinear in t and is left untouched.
#
#   3. BatchNorm folded into the preceding conv, plus channels_last, which is
#      what oneDNN wants for these small 24x24 feature maps.
#
# All three are algebraic identities, not approximations -- check_fast_engine()
# verifies the output against the reference forward() to float32 round-off.
# ============================================================

import torch.nn.utils.fusion as _fusion


def fold_bn_eval(module):
    """Fold every Conv2d->BatchNorm2d pair inside nn.Sequential containers."""
    for name, child in module.named_children():
        if isinstance(child, nn.Sequential):
            ch, new, i = list(child.children()), [], 0
            while i < len(ch):
                if (i + 1 < len(ch)
                        and isinstance(ch[i], nn.Conv2d)
                        and isinstance(ch[i + 1], nn.BatchNorm2d)):
                    new.append(_fusion.fuse_conv_bn_eval(ch[i], ch[i + 1]))
                    i += 2
                else:
                    new.append(ch[i])
                    i += 1
            setattr(module, name, nn.Sequential(*new))
        else:
            fold_bn_eval(child)
    return module


class FastSliceUNet:
    """
    Exact, faster evaluation of UnifiedSliceUNet for whole-axis inference.

    Works on a BN-folded, channels_last copy of the trained model. The public
    entry point is predict_axis(); the three stages are exposed separately so
    they can be profiled.
    """

    def __init__(self, model, H, W):
        self.m = model
        self.H, self.W = H, W

        f, td = model.base_width, model.time_dim
        conv0 = model.fuse[0]
        w = conv0.weight.detach()                      # [f, f + td, 3, 3]

        self.Wf = w[:, :f].contiguous()                # feat half of the kernel
        # a folded conv+BN carries a bias; it belongs with the feat half
        self.bf = None if conv0.bias is None else conv0.bias.detach().contiguous()
        Wt = w[:, f:].contiguous()                     # time half

        # Response of the time half to a spatially constant map. Zero padding
        # makes this depend on (y, x) only through the border class, so 9
        # representative pixels capture every distinct value.
        dev, dt = w.device, w.dtype
        basis = torch.eye(td, device=dev, dtype=dt).view(td, td, 1, 1)
        basis = basis.expand(td, td, H, W).contiguous()
        K = F.conv2d(basis, Wt, padding=1)             # [td, f, H, W]

        ys, xs = (0, H // 2, H - 1), (0, W // 2, W - 1)
        self.R = torch.stack([K[:, :, y, x] for y in ys for x in xs], 0)  # [9,td,f]

        ry = torch.ones(H, dtype=torch.long, device=dev); ry[0] = 0; ry[-1] = 2
        rx = torch.ones(W, dtype=torch.long, device=dev); rx[0] = 0; rx[-1] = 2
        self.region = (3 * ry[:, None] + rx[None, :]).reshape(-1)         # [H*W]

        self.tail = nn.Sequential(*list(model.fuse.children())[1:])

    # ---------- stage 1 : encoder, once per stored slice ----------
    @torch.inference_mode()
    def encode_axis(self, X_axis, chunk=8):
        """X_axis: [S, Cin, H, W] (normalised) -> [S, f, H, W]"""
        out = [self.m.encode_one_slice(X_axis[i:i + chunk])
               for i in range(0, X_axis.shape[0], chunk)]
        return torch.cat(out, 0)

    # ---------- stage 2 : window gather in feature space ----------
    @torch.inference_mode()
    def fuse_windows(self, feats, win_ids):
        """feats: [S, f, H, W]; win_ids: [B, Wn] -> [B, f, H, W]"""
        B, Wn = win_ids.shape
        S, f, H, W = feats.shape
        g = feats[win_ids.reshape(-1)].reshape(B, Wn * f, H, W)
        return self.m.slice_fuse(g.contiguous(memory_format=torch.channels_last))

    # ---------- stage 3 : time head ----------
    @torch.inference_mode()
    def time_head(self, feat, times, chunk=4):
        """feat: [B, f, H, W]; times: [B, T] (normalised) -> [B, T, H, W]"""
        B, f, H, W = feat.shape
        T = times.shape[1]

        t_emb = self.m.time_mlp(
            time_fourier_features(times, n_freq=self.m.time_n_freq))   # [B,T,td]

        # time contribution to fuse[0], without ever materialising a constant
        # [B*T, td, H, W] map
        P = torch.einsum('btc,rco->rbto', t_emb, self.R)               # [9,B,T,f]
        term = P.permute(1, 2, 3, 0)[..., self.region].reshape(B, T, f, H, W)

        u = F.conv2d(feat, self.Wf, bias=self.bf, padding=1)           # [B,f,H,W]

        outs = []
        for i in range(0, B, chunk):
            z = u[i:i + chunk, None] + term[i:i + chunk]
            b = z.shape[0]
            z = z.reshape(b * T, f, H, W).contiguous(memory_format=torch.channels_last)
            outs.append(self.tail(z).reshape(b, T, H, W))
        return torch.cat(outs, 0)

    @torch.inference_mode()
    def predict_axis(self, X_axis, win_ids, times):
        feats = self.encode_axis(X_axis)
        feat = self.fuse_windows(feats, win_ids)
        return self.time_head(feat, times)


# ------------------------------------------------------------
# ENGINE BUILD + THREAD AUTO-TUNE
#
# torch's default (one thread per logical core) is usually NOT the fastest here:
# the feature maps are 24x24, so oneDNN spends more time on thread hand-off than
# on arithmetic. A 5-second sweep picks the best count for this machine.
# ------------------------------------------------------------
def autotune_threads(fast, X_axis, win_ids, times, candidates=None, verbose=True):
    n = os.cpu_count() or 4
    if candidates is None:
        candidates = sorted({2, 4, n // 4 or 1, n // 2 or 1, n})
    probe = slice(0, min(4, win_ids.shape[0]))
    best, best_t = torch.get_num_threads(), float("inf")
    for th in candidates:
        torch.set_num_threads(th)
        fast.predict_axis(X_axis, win_ids[probe], times[probe])        # warm
        t0 = time.perf_counter()
        fast.predict_axis(X_axis, win_ids[probe], times[probe])
        dt = time.perf_counter() - t0
        if verbose:
            print(f"    threads={th:3d} -> {1e3 * dt:7.1f} ms (probe)")
        if dt < best_t:
            best, best_t = th, dt
    torch.set_num_threads(best)
    if verbose:
        print(f"    using {best} threads")
    return best


def build_fast_engine(bundle, threads=None, autotune=True, verbose=True):
    """
    Build the optimised inference engine for a loaded bundle.

    threads=None + autotune=True -> sweep and pick. Pass an int to force one.
    """
    ds = bundle["ds"]
    H, W = ds.X_by_axis["x"].shape[-2:]

    model = copy.deepcopy(bundle["model"]).eval()
    model = fold_bn_eval(model).to(memory_format=torch.channels_last)
    fast = FastSliceUNet(model, H, W)

    x_mean = ds.x_mean.astype(np.float32)          # [Cin,1,1]
    x_std = ds.x_std.astype(np.float32)

    # window index table per axis (same clamp/reflect rule as the dataset)
    win_ids = {}
    for axis in ("x", "y", "z"):
        S = ds.X_by_axis[axis].shape[1]
        win_ids[axis] = torch.tensor(
            [ds._get_window_ids(c, S) for c in range(S)],
            dtype=torch.long, device=DEVICE)

    times_norm = torch.from_numpy(
        (ds.times / ds.t_scale).astype(np.float32)).to(DEVICE)          # [T]

    engine = {
        "fast": fast,
        "model": model,
        "H": H, "W": W,
        "x_mean": x_mean, "x_std": x_std,
        "win_ids": win_ids,
        "times_norm": times_norm,
        "y_mean": float(ds.y_mean), "y_std": float(ds.y_std),
        "target_idx": int(ds.target_idx),
    }

    if threads is not None:
        torch.set_num_threads(int(threads))
        engine["threads"] = int(threads)
    elif autotune:
        if verbose:
            print("  autotuning CPU threads...")
        case0 = int(bundle["split_ids"][0])
        Xa = _axis_input(bundle, engine, case0, "x")
        S = Xa.shape[0]
        engine["threads"] = autotune_threads(
            fast, Xa, win_ids["x"],
            times_norm[None].expand(S, -1), verbose=verbose)
    else:
        engine["threads"] = torch.get_num_threads()

    if verbose:
        print(f"  fast engine ready | {H}x{W} | window={bundle['window_size']} "
              f"| threads={engine['threads']} | device={DEVICE}")
    return engine


def _axis_input(bundle, engine, case_id, axis):
    """Normalised static input for every stored slice of one case+axis."""
    Xa = bundle["ds"].X_by_axis[axis][int(case_id)]        # [S, Cin, H, W]
    Xa = (np.asarray(Xa, dtype=np.float32) - engine["x_mean"]) / engine["x_std"]
    return (torch.from_numpy(Xa)
            .to(DEVICE)
            .contiguous(memory_format=torch.channels_last))


# ------------------------------------------------------------
# CORRECTNESS CHECK
# ------------------------------------------------------------
@torch.inference_mode()
def check_fast_engine(bundle, engine, case_id=None, n_slices=3, axis="x"):
    """Compare the fast engine against the reference forward() on a few slices."""
    if case_id is None:
        case_id = int(bundle["split_ids"][0])

    ds = bundle["ds"]
    S = ds.X_by_axis[axis].shape[1]
    n = min(n_slices, S)

    idxs = [i for i, s in enumerate(ds.samples)
            if int(s["case_id"]) == int(case_id) and s["axis_name"] == axis]
    idxs = idxs[:n]

    Xw = torch.stack([ds[i]["X_window"] for i in idxs]).to(DEVICE)
    Tt = torch.stack([ds[i]["times"] for i in idxs]).to(DEVICE)
    ref = bundle["model"](Xw, Tt)

    Xa = _axis_input(bundle, engine, case_id, axis)
    new = engine["fast"].predict_axis(Xa, engine["win_ids"][axis][:n], Tt)

    d = float((ref - new).abs().max())
    rng = float(ref.abs().max())
    print(f"fast-engine check | max|diff| = {d:.3e}   (field scale {rng:.3f})   "
          f"{'OK' if d < 1e-3 * max(rng, 1.0) else 'MISMATCH'}")
    return d


# ------------------------------------------------------------
# WHOLE-CASE PREDICTION + TIMING  (no plotting inside the timed region)
# ------------------------------------------------------------
@torch.inference_mode()
def predict_case_fast(bundle, engine, case_id, with_truth=True, verbose=True):
    """
    Predict every stored slice of one case, all 3 axes, de-normalised.

    Returns (fields, timing) with the same layout as predict_case_fields():
        fields[axis] = {"pred", "true", "positions", "rmse"}

    timing splits host-side data prep from the model forward, so the number you
    quote against a CFD run is unambiguous.
    """
    ds = bundle["ds"]
    fast = engine["fast"]
    y_mean, y_std = engine["y_mean"], engine["y_std"]
    ti = engine["target_idx"]

    fields = {}
    t_prep = t_fwd = 0.0
    _sync()

    for axis in ("x", "y", "z"):
        t0 = time.perf_counter()
        Xa = _axis_input(bundle, engine, case_id, axis)
        S = Xa.shape[0]
        times = engine["times_norm"][None].expand(S, -1)
        _sync()
        t_prep += time.perf_counter() - t0

        t0 = time.perf_counter()
        pred = fast.predict_axis(Xa, engine["win_ids"][axis], times)
        _sync()
        t_fwd += time.perf_counter() - t0

        pred = pred.cpu().numpy() * y_std + y_mean              # [S, T, H, W]

        entry = {"pred": pred, "positions": list(range(S))}
        if with_truth:
            true = np.asarray(
                ds.Y_by_axis[axis][int(case_id), :, :, ti], dtype=np.float32)
            entry["true"] = true

            # whole-axis accuracy
            entry["parts"] = _r2_parts(true, pred)
            entry["rmse"] = float(np.sqrt(np.mean((pred - true) ** 2)))
            entry["r2"] = r2_score_np(true, pred)

            # per-slice accuracy, each over that slice's full time sequence
            entry["rmse_per_slice"] = np.sqrt(
                ((pred - true) ** 2).reshape(S, -1).mean(axis=1))
            entry["r2_per_slice"] = np.array(
                [r2_score_np(true[s], pred[s]) for s in range(S)])
        fields[axis] = entry

    n_slices = sum(v["pred"].shape[0] for v in fields.values())
    n_times = int(engine["times_norm"].shape[0])

    timing = {
        "case_id": int(case_id),
        "n_slices": int(n_slices),
        "n_times": n_times,
        "device": str(DEVICE),
        "threads": int(engine.get("threads", torch.get_num_threads())),
        "t_dataprep_s": t_prep,
        "t_forward_s": t_fwd,
        "t_case_total_s": t_prep + t_fwd,
        "t_per_slice_ms": 1e3 * t_fwd / max(n_slices, 1),
        "t_per_field_ms": 1e3 * t_fwd / max(n_slices * n_times, 1),
        "slices_per_s": n_slices / t_fwd if t_fwd > 0 else float("nan"),
        "fields_per_s": (n_slices * n_times) / t_fwd if t_fwd > 0 else float("nan"),
    }

    if verbose:
        print(f"case {case_id}  |  {n_slices} slices x {n_times} snapshots "
              f"= {n_slices * n_times} 2D fields  |  {DEVICE}, "
              f"{timing['threads']} threads")
        print(f"  INFERENCE (no plotting) : forward {1e3 * t_fwd:8.1f} ms"
              f"  + data prep {1e3 * t_prep:7.1f} ms"
              f"  = {1e3 * timing['t_case_total_s']:8.1f} ms")
        print(f"  per slice {timing['t_per_slice_ms']:7.2f} ms"
              f"   | per 2D field {timing['t_per_field_ms']:6.3f} ms"
              f"   | {timing['fields_per_s']:.0f} fields/s")
        if with_truth:
            for axis, v in fields.items():
                print(f"    axis {axis}: RMSE({bundle['target']}) = {v['rmse']:.5f}"
                      f"   R2 = {v['r2']:.5f}")

    return fields, timing


@torch.inference_mode()
def benchmark_case_fast(bundle, engine, case_id, n_warmup=1, n_repeat=3,
                        verbose=True):
    """
    Timing only: median of n_repeat whole-case forward sweeps, warm caches.
    Data prep is done once up front and excluded, so this is pure model cost.
    """
    fast = engine["fast"]
    packs = []
    t0 = time.perf_counter()
    for axis in ("x", "y", "z"):
        Xa = _axis_input(bundle, engine, case_id, axis)
        S = Xa.shape[0]
        packs.append((Xa, engine["win_ids"][axis],
                      engine["times_norm"][None].expand(S, -1)))
    t_prep = time.perf_counter() - t0

    for _ in range(n_warmup):
        for p in packs:
            fast.predict_axis(*p)
    _sync()

    runs = []
    for _ in range(n_repeat):
        _sync()
        t0 = time.perf_counter()
        for p in packs:
            fast.predict_axis(*p)
        _sync()
        runs.append(time.perf_counter() - t0)

    t_fwd = float(np.median(runs))
    n_slices = sum(p[0].shape[0] for p in packs)
    n_times = int(engine["times_norm"].shape[0])

    res = {
        "case_id": int(case_id),
        "n_slices": n_slices,
        "n_times": n_times,
        "device": str(DEVICE),
        "threads": int(engine.get("threads", torch.get_num_threads())),
        "t_forward_s": t_fwd,
        "t_forward_std_s": float(np.std(runs)),
        "t_dataprep_s": t_prep,
        "t_case_total_s": t_fwd + t_prep,
        "t_per_slice_ms": 1e3 * t_fwd / n_slices,
        "t_per_field_ms": 1e3 * t_fwd / (n_slices * n_times),
        "fields_per_s": (n_slices * n_times) / t_fwd,
    }

    if verbose:
        print(f"case {case_id}: forward {1e3 * t_fwd:.1f} +/- "
              f"{1e3 * res['t_forward_std_s']:.1f} ms "
              f"({n_repeat} runs, {n_slices} slices x {n_times} snapshots)")
    return res


# ------------------------------------------------------------
# SIMPLE TOTAL INFERENCE TIME
#
# One number: how long the model takes to produce the whole matrix of predicted
# images for one case. No figures, no colorbars, nothing but the model.
# ------------------------------------------------------------
@torch.inference_mode()
def time_case_inference(bundle, engine, case_index=0, repeat=1,
                        with_rmse=True, return_pred=False, verbose=True):
    """
    Time the prediction of the full [slices x snapshots] image matrix for ONE case.

    Returns a dict; add return_pred=True to also get the predictions themselves
    as fields[axis]["pred"] with shape [S, T, H, W].
    """
    case_id = int(bundle["split_ids"][int(case_index)])

    runs, fields = [], None
    for _ in range(max(1, int(repeat))):
        fields, t = predict_case_fast(bundle, engine, case_id,
                                      with_truth=with_rmse, verbose=False)
        runs.append(t)

    t_fwd = float(np.median([r["t_forward_s"] for r in runs]))
    t_prep = float(np.median([r["t_dataprep_s"] for r in runs]))
    t_tot = t_fwd + t_prep

    n_slices = runs[-1]["n_slices"]
    n_times = runs[-1]["n_times"]
    n_img = n_slices * n_times
    H, W = engine["H"], engine["W"]

    res = {
        "case_id": case_id,
        "n_images": n_img,
        "image_shape": (H, W),
        "n_slices": n_slices,
        "n_times": n_times,
        "t_total_s": t_tot,
        "t_forward_s": t_fwd,
        "t_dataprep_s": t_prep,
        "ms_per_image": 1e3 * t_tot / n_img,
        "ms_per_slice": 1e3 * t_tot / n_slices,
        "device": str(DEVICE),
        "threads": int(engine.get("threads", torch.get_num_threads())),
    }
    if with_rmse:
        # pooled over every axis, slice and snapshot of the case
        res["rmse"], res["r2"] = _pool_r2([v["parts"] for v in fields.values()])
        res["r2_per_axis"] = {a: float(v["r2"]) for a, v in fields.items()}

    if verbose:
        print(f"case {case_id}  |  {n_slices} slices x {n_times} snapshots "
              f"= {n_img} predicted images of {H}x{W}")
        print(f"TOTAL INFERENCE TIME : {t_tot:.2f} s"
              f"   ({res['ms_per_image']:.1f} ms per image,"
              f" {res['ms_per_slice']:.0f} ms per slice)")
        print(f"  forward {t_fwd:.2f} s | data prep {t_prep:.2f} s"
              f" | {res['device']}, {res['threads']} threads"
              + (f" | RMSE({bundle['target']}) = {res['rmse']:.4f}"
                 f" | TOTAL R2 = {res['r2']:.4f}" if with_rmse else ""))
        if with_rmse:
            for a, v in fields.items():
                print(f"  axis {a}: RMSE = {v['rmse']:.5f}   R2 = {v['r2']:.5f}"
                      f"   ({v['pred'].shape[0]} slices)")

    return (res, fields) if return_pred else res


# ------------------------------------------------------------
# PER-SLICE ACCURACY TABLE
# ------------------------------------------------------------
def slice_metrics_table(fields, sort_by=None):
    """
    One row per stored slice: RMSE and R2 over that slice's full time sequence.

    `fields` is what predict_case_fast()/time_case_inference(return_pred=True)
    returns. Pass sort_by="r2" to see the worst planes first.
    """
    rows = []
    for axis, v in fields.items():
        if "r2_per_slice" not in v:
            continue
        for s in range(v["pred"].shape[0]):
            rows.append({
                "axis": axis,
                "slice": int(v["positions"][s]),
                "rmse": float(v["rmse_per_slice"][s]),
                "r2": float(v["r2_per_slice"][s]),
            })
    df = pd.DataFrame(rows)
    if sort_by is not None and not df.empty:
        df = df.sort_values(sort_by).reset_index(drop=True)
    return df


In [ ]:
# ============================================================
# PER-SLICE INFERENCE PLOT  (updated)
#
# Fixes over the previous version:
#   * Q is DE-NORMALISED before plotting. dataset.__getitem__ returns
#     (X - x_mean)/x_std, so raw channel 0 is not in W/m^3.
#   * Row 1 gets a colorbar and carries its own [min, max] in the title.
#     A plane that misses the battery is a constant array, which gives
#     vmin == vmax and renders as a flat white panel - that is what made the
#     old Q row look empty. Now it is labelled "uniform / no battery cut".
#   * The battery cross-section (input channel "bat_mask") is outlined on the
#     Q, truth and prediction panels whenever the plane cuts the prism.
#   * The forward pass is timed and the timing is returned.
# ============================================================

def _channel_index(meta, name, default):
    """Look a channel up by name in meta['channel_order'], else fall back."""
    try:
        order = list(meta.get("channel_order", []))
        return order.index(name)
    except Exception:
        return default


def _sync_device():
    if torch.cuda.is_available():
        torch.cuda.synchronize()


@torch.no_grad()
def plot_slice_timeseries(
    model,
    dataset,
    stats,
    case_id,
    axis="x",
    slice_idx=0,
    ncols=5,
    heat_ch=0,
    bat_ch=None,
    meta=None,
    crop_padding=True,
    show_battery_outline=True,
    q_log=True,
    verbose=True,
    pred_override=None,
):
    """
    Per-slice inference plot.

      row 0 : the 6 boundary-condition face images
      row 1 : heat source Q on this plane, in PHYSICAL units, with colorbar
      row 2 : ground truth
      row 3 : prediction
      row 4 : |error|

    Returns a dict with the Q range, whether the plane cuts the battery, and
    the forward-pass time for this single slice.
    """

    model.eval()

    # --------------------------------------------------------
    # find sample
    # --------------------------------------------------------
    match_idx = None
    for i, s in enumerate(dataset.samples):
        if isinstance(s, dict):
            s_case  = s.get("case_id", s.get("case_index"))
            s_axis  = s.get("axis_name")
            s_slice = s.get("slice_idx", s.get("slice_index"))
        else:
            s_case, s_axis, s_slice = s

        if int(s_case) == int(case_id) and s_axis == axis and int(s_slice) == int(slice_idx):
            match_idx = i
            break

    if match_idx is None:
        raise ValueError(f"No sample found for case_id={case_id}, axis='{axis}', slice_idx={slice_idx}")

    sample = dataset[match_idx]

    # --------------------------------------------------------
    # input + timed forward pass
    # --------------------------------------------------------
    X_window = sample["X_window"][None].to(DEVICE)   # [1,Wn,Cin,H,W]
    times_in = sample["times"][None].to(DEVICE)      # [1,T]

    Y_true = sample["Y_raw"].cpu().numpy()           # [T,H,W]

    if pred_override is not None:
        # already de-normalised, reused from the batched whole-case sweep so
        # that no slice is pushed through the model twice
        pred = np.asarray(pred_override, dtype=np.float32)
        t_forward = float("nan")
    else:
        _sync_device()
        _t0 = time.perf_counter()
        pred = model(X_window, times_in)
        _sync_device()
        t_forward = time.perf_counter() - _t0

        pred = pred[0].cpu().numpy()
        pred = pred * stats["y_std"] + stats["y_mean"]

    t_note = ("reused from batched sweep" if pred_override is not None
              else f"forward {1e3 * t_forward:.1f} ms")

    err = np.abs(pred - Y_true)

    # --------------------------------------------------------
    # centre input slice (still NORMALISED at this point)
    # --------------------------------------------------------
    X_window_np = sample["X_window"].cpu().numpy()   # [Wn,Cin,H,W]
    center_id = X_window_np.shape[0] // 2
    X_center = X_window_np[center_id]                # [Cin,H,W]

    # --------------------------------------------------------
    # time selection
    # --------------------------------------------------------
    Tn = Y_true.shape[0]
    idxs = np.linspace(0, Tn - 1, min(ncols, Tn)).round().astype(int)

    if "times_raw" in sample:
        times_phys = sample["times_raw"].cpu().numpy()
    else:
        times_phys = sample["times"].cpu().numpy()

    # --------------------------------------------------------
    # DE-NORMALISE the input channels we want to look at
    # --------------------------------------------------------
    if not (0 <= heat_ch < X_center.shape[0]):
        raise ValueError(f"heat_ch={heat_ch} out of range")

    x_mean = np.asarray(stats["x_mean"]).reshape(-1)
    x_std  = np.asarray(stats["x_std"]).reshape(-1)

    Qmap = X_center[heat_ch] * x_std[heat_ch] + x_mean[heat_ch]      # W/m^3

    if bat_ch is None:
        bat_ch = _channel_index(meta or {}, "bat_mask", 16)

    bat_map = None
    if show_battery_outline and 0 <= bat_ch < X_center.shape[0]:
        bat_map = X_center[bat_ch] * x_std[bat_ch] + x_mean[bat_ch]
        bat_map = (bat_map > 0.5).astype(np.float32)

    # --------------------------------------------------------
    # BC faces from actual channels
    # --------------------------------------------------------
    if X_center.shape[0] < 10:
        raise ValueError(f"Expected at least 10 channels, got {X_center.shape[0]}")

    bc_faces = {
        "Left":   X_center[4] * x_std[4] + x_mean[4],
        "Right":  X_center[5] * x_std[5] + x_mean[5],
        "Front":  X_center[6] * x_std[6] + x_mean[6],
        "Back":   X_center[7] * x_std[7] + x_mean[7],
        "Bottom": X_center[8] * x_std[8] + x_mean[8],
        "Top":    X_center[9] * x_std[9] + x_mean[9],
    }
    bc_order = ["Left", "Right", "Front", "Back", "Bottom", "Top"]

    # --------------------------------------------------------
    # crop padding
    # --------------------------------------------------------
    pad_h, pad_w = 0, 0
    if crop_padding and meta is not None:
        pad_h, pad_w = get_axis_pad(meta, axis)

        Qmap   = unpad_2d(Qmap, pad_h, pad_w)
        Y_true = np.stack([unpad_2d(y, pad_h, pad_w) for y in Y_true], axis=0)
        pred   = np.stack([unpad_2d(y, pad_h, pad_w) for y in pred], axis=0)
        err    = np.stack([unpad_2d(y, pad_h, pad_w) for y in err], axis=0)

        if bat_map is not None:
            bat_map = unpad_2d(bat_map, pad_h, pad_w)

        for kf in bc_faces:
            bc_faces[kf] = unpad_2d(bc_faces[kf], pad_h, pad_w)

    # --------------------------------------------------------
    # per-slice accuracy, over the whole time sequence of this plane
    # (computed after cropping, so it matches exactly what is drawn)
    # --------------------------------------------------------
    slice_rmse = float(np.sqrt(np.mean((pred - Y_true) ** 2)))
    slice_r2 = r2_score_np(Y_true, pred)

    # --------------------------------------------------------
    # Q panel scaling
    # --------------------------------------------------------
    q_min, q_max = float(Qmap.min()), float(Qmap.max())
    q_flat = (q_max - q_min) <= 1e-9 * max(abs(q_max), 1.0)

    cuts_battery = bool(bat_map is not None and bat_map.any())

    q_norm = None
    if q_log and not q_flat and q_min > 0.0:
        from matplotlib.colors import LogNorm
        q_norm = LogNorm(vmin=max(q_min, 1e-3), vmax=q_max)

    if q_flat:
        q_note = f"uniform {q_min:.4g} W/m$^3$ (no battery cut)"
    elif cuts_battery:
        q_note = f"[{q_min:.3g}, {q_max:.3g}] W/m$^3$ - cuts battery"
    else:
        q_note = f"[{q_min:.3g}, {q_max:.3g}] W/m$^3$ - background only"

    # --------------------------------------------------------
    # target colour range shared across time, so the melting front is
    # comparable between columns and between truth and prediction
    # --------------------------------------------------------
    # Use the dataset's own target, not the notebook-level TARGET global: a
    # checkpoint trained on "T" can be loaded while TARGET is still "f".
    tgt = getattr(dataset, "target", TARGET)

    if tgt == "f":
        y_vmin, y_vmax = 0.0, 1.0
    else:
        y_vmin = float(min(Y_true.min(), pred.min()))
        y_vmax = float(max(Y_true.max(), pred.max()))

    def _outline(ax):
        if show_battery_outline and cuts_battery:
            ax.contour(bat_map, levels=[0.5], colors="cyan", linewidths=1.2)

    # --------------------------------------------------------
    # plotting
    # --------------------------------------------------------
    fig, axs = plt.subplots(5, len(idxs), figsize=(3.1 * len(idxs), 13))

    if len(idxs) == 1:
        axs = np.array(axs).reshape(5, 1)

    # row 0: BC faces
    for j in range(len(idxs)):
        axs[0, j].axis("off")

    for j, face_name in enumerate(bc_order[:len(idxs)]):
        imb = axs[0, j].imshow(bc_faces[face_name], origin="lower", cmap="viridis")
        axs[0, j].set_title(face_name, fontsize=9)
        fig.colorbar(imb, ax=axs[0, j], shrink=0.75)
        axs[0, j].axis("off")

    # remaining rows
    for j, ti in enumerate(idxs):
        imq = axs[1, j].imshow(Qmap, origin="lower", cmap="Reds",
                               norm=q_norm,
                               vmin=None if q_norm is not None else q_min,
                               vmax=None if q_norm is not None else (q_max if not q_flat else q_min + 1.0))
        axs[1, j].set_title(f"Q  {q_note}", fontsize=8)
        fig.colorbar(imq, ax=axs[1, j], shrink=0.75)
        _outline(axs[1, j])
        axs[1, j].axis("off")

        imt = axs[2, j].imshow(Y_true[ti], origin="lower", cmap="inferno",
                               vmin=y_vmin, vmax=y_vmax)
        axs[2, j].set_title(f"True-{tgt} @ {float(times_phys[ti]):.0f}s", fontsize=9)
        fig.colorbar(imt, ax=axs[2, j], shrink=0.75)
        _outline(axs[2, j])
        axs[2, j].axis("off")

        imp = axs[3, j].imshow(pred[ti], origin="lower", cmap="inferno",
                               vmin=y_vmin, vmax=y_vmax)
        axs[3, j].set_title(f"Pred-{tgt}", fontsize=9)
        fig.colorbar(imp, ax=axs[3, j], shrink=0.75)
        _outline(axs[3, j])
        axs[3, j].axis("off")

        ime = axs[4, j].imshow(err[ti], origin="lower", cmap="magma")
        axs[4, j].set_title(f"|error|  max {float(err[ti].max()):.3g}", fontsize=8)
        fig.colorbar(ime, ax=axs[4, j], shrink=0.75)
        axs[4, j].axis("off")

    fig.suptitle(
        f"case {case_id} | axis {axis} | stored slice {slice_idx}"
        f" | {'battery in plane' if cuts_battery else 'battery NOT in plane'}"
        f" | RMSE {slice_rmse:.4f} | R$^2$ {slice_r2:.4f}"
        f" | {t_note}",
        fontsize=11,
    )
    plt.tight_layout()
    plt.show()

    if verbose:
        print(f"  Q range        : [{q_min:.4g}, {q_max:.4g}] W/m^3"
              f"{'  (constant)' if q_flat else ''}")
        print(f"  battery in cut : {cuts_battery}")
        print(f"  RMSE ({tgt})     : {slice_rmse:.5f}")
        print(f"  R2   ({tgt})     : {slice_r2:.5f}")
        print(f"  forward time   : {t_note} "
              f"({Tn} time snapshots in one pass)")

    return {
        "q_min": q_min,
        "q_max": q_max,
        "q_flat": bool(q_flat),
        "cuts_battery": cuts_battery,
        "t_forward_s": float(t_forward),
        "rmse": slice_rmse,
        "r2": slice_r2,
        "n_times": int(Tn),
    }


In [49]:
@torch.no_grad()
def infer_and_plot_from_ckpt(
    ckpt_path,
    npz_path,
    split="test",
    case_index=0,
    axis="x",
    slice_idx=0,
    ncols=5,
    heat_ch=0,
    bundle=None,
    only_if_cuts_battery=False,
    pred_override=None,
):
    """
    Thin wrapper around the cached loader + plot_slice_timeseries.

    Pass `bundle` to skip the (expensive) reload entirely. `case_index` indexes
    into the split list; the underlying case id is looked up for you.
    """
    if bundle is None:
        bundle = load_model_and_dataset(ckpt_path, npz_path, split=split)

    split_ids = bundle["split_ids"]
    if not (0 <= case_index < len(split_ids)):
        raise IndexError(
            f"case_index={case_index} out of range for split='{split}' "
            f"with {len(split_ids)} cases"
        )

    case_id = int(split_ids[case_index])

    if only_if_cuts_battery:
        if slice_idx not in cutting_slice_positions(bundle, case_id, axis):
            print(f"  skipped: axis {axis} slice {slice_idx} does not cut the battery")
            return bundle, None

    info = plot_slice_timeseries(
        model=bundle["model"],
        dataset=bundle["ds"],
        stats=bundle["stats"],
        case_id=case_id,
        axis=axis,
        slice_idx=slice_idx,
        ncols=ncols,
        heat_ch=heat_ch,
        meta=bundle["meta"],
        crop_padding=True,
        pred_override=pred_override,
    )

    return bundle, info


In [ ]:
# ============================================================
# DRIVER : load once -> build the fast engine -> TOTAL INFERENCE TIME
#
# The first run writes the .npy cache (a few minutes, one-off). Every run after
# that memory-maps it, so the load is milliseconds instead of ~4 minutes.
#
# The timing block runs the model only: no figure is created and no matplotlib
# call is made inside it. Set DO_PLOTS = True for the per-slice panels.
# ============================================================

CKPT_PATH = r"C:\Users\Amar\Downloads\3D tests deeponet battery\dataset_run_20260912-182558\checkpoints_unified_slice_unet_f\best.pt"
NPZ_PATH  = r"C:\Users\Amar\Downloads\3D tests deeponet battery\dataset_run_20260912-182558\slice_dataset_3d.npz"

CASE_INDEX       = 0      # index into the test split
REPEAT           = 1      # >1 to median over several sweeps
DO_PLOTS         = True  # True for the full per-slice visualisation
MAX_PLOTS_PER_AX = 5
NCOLS            = 8

bundle = load_model_and_dataset(CKPT_PATH, NPZ_PATH, split="test")
engine = build_fast_engine(bundle)

# ------------------------------------------------------------
# TOTAL INFERENCE TIME  (visualisation excluded)
# ------------------------------------------------------------
res, fields = time_case_inference(
    bundle, engine,
    case_index=CASE_INDEX,
    repeat=REPEAT,
    return_pred=True,
)

pd.DataFrame([res]).to_csv(Path(NPZ_PATH).parent / "inference_timing.csv", index=False)

# per-slice RMSE / R2 over each plane's full time sequence
df_slices = slice_metrics_table(fields)
display(df_slices)
df_slices.to_csv(Path(NPZ_PATH).parent / "inference_slice_metrics.csv", index=False)
print(f"worst 3 planes by R2:\n"
      f"{slice_metrics_table(fields, sort_by='r2').head(3).to_string(index=False)}")

# fields[axis]["pred"] is the [S, T, H, W] matrix of predicted images, already
# de-normalised and ready to use without re-running the model.

# ------------------------------------------------------------
# OPTIONAL PLOTS (untimed, reuse the predictions computed above)
# ------------------------------------------------------------
if DO_PLOTS:
    PLOT_ALL_SLICES = True
    case_id = res["case_id"]
    cov = slice_coverage_report(bundle, case_id)

    for axis in ["x", "y", "z"]:
        if PLOT_ALL_SLICES:
            positions = list(range(bundle["ds"].X_by_axis[axis].shape[1]))
        else:
            positions = cov[axis][:MAX_PLOTS_PER_AX]

        if not positions:
            print(f"\n===== AXIS {axis.upper()}: no plane cuts the battery "
                  f"for case {case_id} =====")
            continue

        print(f"\n===== AXIS {axis.upper()}  |  plotting stored slices {positions} =====")

        for p in positions:
            infer_and_plot_from_ckpt(
                ckpt_path=CKPT_PATH,
                npz_path=NPZ_PATH,
                split="test",
                case_index=CASE_INDEX,
                axis=axis,
                slice_idx=p,
                ncols=NCOLS,
                heat_ch=0,
                bundle=bundle,
                pred_override=fields[axis]["pred"][p],
            )


In [51]:
print(model)

UnifiedSliceUNet(
  (lift): Conv2d(18, 64, kernel_size=(1, 1), stride=(1, 1))
  (unet1): PaperUNetBlock2D(
    (c1): Sequential(
      (0): Conv2d(64, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
    )
    (c2): Sequential(
      (0): Conv2d(64, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
    )
    (c3): Sequential(
      (0): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
    )
    (c4): Sequential(
      (0): Conv2d(64, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, tra